In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import neattext.functions as nfx
import plotly.express as plx
from sklearn.metrics import classification_report

import tensorflow as tf
from tensorflow.keras.layers import Embedding, Dense, GlobalMaxPooling1D, Input, LSTM
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm

In [3]:
data=pd.read_csv('Suicide_Detection.csv')
data.head()

,Unnamed: 0,text,class
0,2,Ex Wife Threatening SuicideRecently I left my ...,suicide
1,3,Am I weird I don't get affected by compliments...,non-suicide
2,4,Finally 2020 is almost over... So I can never ...,non-suicide
3,8,i need helpjust help me im crying so hard,suicide
4,9,"I’m so lostHello, my name is Adam (16) and I’v...",suicide


In [4]:
data['class'].value_counts()

class
suicide        116037
non-suicide    116037
Name: count, dtype: int64

In [5]:
data['class'].value_counts().index.values

array(['suicide', 'non-suicide'], dtype=object)

In [6]:
train_data,test_data=train_test_split(data,test_size=0.2,random_state=10)

In [7]:
train_data['class'].value_counts().index.values

array(['suicide', 'non-suicide'], dtype=object)

In [8]:
# data cleaning
def clean_text(text):
    text_length=[]
    cleaned_text=[]
    for sent in tqdm(text):
        sent=sent.lower()
        sent=nfx.remove_special_characters(sent)
        sent=nfx.remove_stopwords(sent)
        text_length.append(len(sent.split()))
        cleaned_text.append(sent)
    return cleaned_text,text_length

In [9]:
cleaned_train_text,train_text_length=clean_text(train_data.text)
cleaned_test_text,test_text_length=clean_text(test_data.text)

100%|██████████| 46415/46415 [00:05<00:00, 8177.33it/s] 


In [10]:
tokenizer=Tokenizer()
tokenizer.fit_on_texts(cleaned_train_text)

In [11]:
cleaned_train_text[0]

'hey east cost ya guys doin whats snow like'

In [12]:
train_text_seq=tokenizer.texts_to_sequences(cleaned_train_text)
train_text_pad=pad_sequences(train_text_seq,maxlen=50)

In [13]:
test_text_seq=tokenizer.texts_to_sequences(cleaned_test_text)
test_text_pad=pad_sequences(test_text_seq,maxlen=50)

In [14]:
test_text_seq

[[590,
  2970,
  28775,
  1711,
  26001,
  486,
  147,
  118,
  2387,
  757,
  357,
  68,
  415,
  4690,
  8712,
  1415,
  99,
  364,
  735,
  87407],
 [424,
  1512,
  518,
  65672,
  494,
  3356,
  1293,
  13,
  389,
  255,
  2620,
  41,
  5764,
  1302,
  513,
  1685,
  189,
  2119,
  37234,
  849,
  15965,
  20417,
  54,
  436,
  3429,
  3456,
  93,
  150,
  3],
 [3507, 755, 2186],
 [1,
  107,
  115,
  31,
  10315,
  557,
  200195,
  105,
  17,
  602,
  74,
  493,
  274,
  57,
  96,
  11,
  144,
  977,
  153,
  23,
  14,
  93,
  143,
  460,
  1,
  1057,
  348,
  958,
  1719,
  205,
  48,
  26,
  16,
  8,
  58,
  57,
  61,
  200195,
  105,
  57,
  1092,
  139,
  21,
  54,
  2,
  1704,
  325,
  16199,
  2583,
  146,
  1,
  1278,
  28,
  25,
  658,
  1268,
  1266,
  346,
  57,
  177,
  1,
  3387,
  59,
  345,
  8,
  428,
  3861,
  108],
 [909,
  126,
  8,
  736,
  31516,
  68,
  353,
  8936,
  361,
  70,
  6,
  1576,
  270,
  4,
  36,
  62,
  62,
  62,
  11,
  12,
  26,
  22,
  83,
  85

In [15]:
train_text_pad

array([[   0,    0,    0, ...,  176, 3027,    3],
       [   0,    0,    0, ...,  163,  508, 1642],
       [   0,    0,    0, ...,   77,  240,   96],
       ...,
       [   0,    0,    0, ...,  328,    2,    4],
       [   0,    0,    0, ...,   65,   26,   16],
       [   4,   46,   25, ...,    2,    4,   16]])

In [16]:
# glove embeddings
lbl_target=LabelEncoder()
train_output=lbl_target.fit_transform(train_data['class'])
test_output=lbl_target.transform(test_data['class'])

In [17]:
import pickle
with open('glove.840B.300d.pkl', 'rb') as fp:
    glove_embedding = pickle.load(fp)

In [18]:
v=len(tokenizer.word_index)

embedding_matrix=np.zeros((v+1,300), dtype=float)
for word,idx in tokenizer.word_index.items():
    embedding_vector=glove_embedding.get(word)
    if embedding_vector is not None:
        embedding_matrix[idx]=embedding_vector

In [19]:
embedding_matrix

array([[ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        ,  0.        ],
       [ 0.074482  ,  0.58293003, -0.78233999, ..., -0.24984001,
        -0.096953  ,  0.66692001],
       [-0.35394999,  0.23051   , -0.62689   , ..., -0.20720001,
         0.52003002,  0.51129001],
       ...,
       [ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        ,  0.        ],
       [ 0.29547   , -0.21822999, -0.039817  , ...,  0.62642998,
         0.48798001, -0.47554001],
       [ 0.75085002, -0.35099   ,  0.37674999, ..., -0.066863  ,
         0.79632998, -0.05967   ]])

In [20]:
early_stop=EarlyStopping(patience=5)
reducelr=ReduceLROnPlateau(patience=3)

In [25]:
# Keras Sequential Model Construction
model = Sequential()
model.add(Input(shape=(50,)))
model.add(Embedding(v + 1, 300, weights=[embedding_matrix], trainable=False))
model.add(LSTM(64, return_sequences=True))
model.add(GlobalMaxPooling1D())
model.add(Dense(64, activation="relu"))
model.add(Dense(1, activation="sigmoid"))
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

In [26]:
model.summary()

Model: "sequential_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding_2 (Embedding)     (None, 50, 300)           81560700  
                                                                 
 lstm_2 (LSTM)               (None, 50, 64)            93440     
                                                                 
 global_max_pooling1d_2 (Gl  (None, 64)                0         
 obalMaxPooling1D)                                               
                                                                 
 dense_4 (Dense)             (None, 64)                4160      
                                                                 
 dense_5 (Dense)             (None, 1)                 65        
                                                                 
Total params: 81658365 (311.50 MB)
Trainable params: 97665 (381.50 KB)
Non-trainable params: 81560700 (311.13 MB)
______

In [27]:
# Model Training and Evaluation
r=model.fit(train_text_pad,train_output,validation_data=(test_text_pad,test_output),
            epochs=20,batch_size=256,callbacks=[early_stop,reducelr])

Epoch 1/20


726/726 [==============================] - 170s 224ms/step - loss: 0.2311 - accuracy: 0.9078 - val_loss: 0.1968 - val_accuracy: 0.9235 - lr: 0.0010
Epoch 2/20
726/726 [==============================] - 224s 308ms/step - loss: 0.1632 - accuracy: 0.9376 - val_loss: 0.1737 - val_accuracy: 0.9317 - lr: 0.0010
Epoch 3/20
726/726 [==============================] - 242s 333ms/step - loss: 0.1389 - accuracy: 0.9471 - val_loss: 0.1669 - val_accuracy: 0.9365 - lr: 0.0010
Epoch 4/20
726/726 [==============================] - 263s 363ms/step - loss: 0.1252 - accuracy: 0.9525 - val_loss: 0.1644 - val_accuracy: 0.9370 - lr: 0.0010
Epoch 5/20
726/726 [==============================] - 211s 290ms/step - loss: 0.1126 - accuracy: 0.9582 - val_loss: 0.1670 - val_accuracy: 0.9383 - lr: 0.0010
Epoch 6/20
726/726 [==============================] - 280s 386ms/step - loss: 0.1020 - accuracy: 0.9622 - val_loss: 0.1728 - val_accuracy: 0.9354 - lr: 0.0010
Epoch 7/20
726/726 [========================

In [29]:
print('TESTING DATA CLASSIFICATION REPORT \n \n')

test_pred = (model.predict(test_text_pad) > 0.5).astype("int32")

print(classification_report(
    test_output,
    test_pred,
    target_names=lbl_target.inverse_transform([0,1])
))


print('TRAINING DATA CLASSIFICATION REPORT \n \n')

train_pred = (model.predict(train_text_pad) > 0.5).astype("int32")

print(classification_report(
    train_output,
    train_pred,
    target_names=lbl_target.inverse_transform([0,1])
))

TESTING DATA CLASSIFICATION REPORT 
 

1451/1451 [==============================] - 21s 13ms/step
              precision    recall  f1-score   support

 non-suicide       0.93      0.95      0.94     23209
     suicide       0.95      0.92      0.93     23206

    accuracy                           0.94     46415
   macro avg       0.94      0.94      0.94     46415
weighted avg       0.94      0.94      0.94     46415

TRAINING DATA CLASSIFICATION REPORT 
 

5802/5802 [==============================] - 128s 22ms/step
              precision    recall  f1-score   support

 non-suicide       0.98      0.98      0.98     92828
     suicide       0.98      0.98      0.98     92831

    accuracy                           0.98    185659
   macro avg       0.98      0.98      0.98    185659
weighted avg       0.98      0.98      0.98    185659



In [39]:
twt = ["it's never too late to be great"]
twt = tokenizer.texts_to_sequences(twt)
twt = pad_sequences(twt, maxlen=50)

prediction = model.predict(twt)[0][0]
print(prediction)

if(prediction > 0.5):
    print("Potential Suicide Post")
else:
    print("Non Suicide Post")

1/1 [==============================] - 0s 56ms/step
0.17836484
Non Suicide Post


In [36]:
pickle.dump(tokenizer, open('tokenizer.pkl', 'wb'))

In [40]:
model.save("model.h5")

c:\Users\user\anaconda3\envs\sd_env\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [41]:
token_form = pickle.load(open('tokenizer.pkl', 'rb'))

In [42]:
from tensorflow.keras.models import load_model

In [43]:

model_form = load_model("model.h5")

In [44]:
twt = ['Through these past years thoughts of suicide, fear, anxiety I’m so close to my limit']
twt = token_form.texts_to_sequences(twt)
twt = pad_sequences(twt, maxlen=50)


prediction = model_form.predict(twt)[0][0]
print(prediction)

if(prediction > 0.5):
    print("Potential Suicide Post")
elif (prediction == 1):
    print("Non Suicide Post")

1/1 [==============================] - 2s 2s/step
0.9184934
Potential Suicide Post
